# Notebook 1 — LoRA Fine-Tuning
Teaches SD 1.5 what spleen ultrasound **looks like** (grayscale texture, speckle noise, acoustic shadows).

**Runtime:** ~40 minutes on A100

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE      = "/content/drive/MyDrive/spleen_project"
LORA_DATA = f"{BASE}/lora/train_data/10_ultrasound"  # "10_" prefix = 10 repeats per image
LORA_OUT  = f"{BASE}/lora/output"

os.makedirs(LORA_DATA, exist_ok=True)
os.makedirs(LORA_OUT,  exist_ok=True)
os.makedirs(f"{BASE}/lora/logs", exist_ok=True)

print("Drive mounted. Folders ready.")

## Cell 2 — Install kohya_ss
The LoRA training library. Takes 3–5 minutes. Ignore the `rich` version warning.

In [ ]:
%%bash
cd /content
git clone -q https://github.com/kohya-ss/sd-scripts
cd sd-scripts
pip install -q -r requirements.txt
pip install -q bitsandbytes xformers wandb
python -c "import torch; print(torch.__version__)"
python -c "import bitsandbytes; print('bitsandbytes ok')"

## Cell 3 — Prepare training images
Converts raw ultrasound PNGs to RGB 512×512 and writes a caption `.txt` file for each image.
kohya_ss requires paired `image.png` + `image.txt` files.

In [ ]:
from PIL import Image
import os

BASE    = "/content/drive/MyDrive/spleen_project"
SRC_DIR = f"{BASE}/raw_images"
OUT_DIR = f"{BASE}/lora/train_data/10_ultrasound"
SIZE    = 512
CAPTION = "ultrasound image"  # same caption for all images

src_files = sorted([f for f in os.listdir(SRC_DIR) if f.endswith(".png")])
print(f"Found {len(src_files)} source images")

for i, fname in enumerate(src_files):
    src_path = os.path.join(SRC_DIR, fname)
    out_img  = os.path.join(OUT_DIR, f"{i:04d}.png")
    out_txt  = os.path.join(OUT_DIR, f"{i:04d}.txt")

    # SD 1.5 expects 3-channel RGB — convert grayscale by duplicating the channel
    img = Image.open(src_path).convert("L")
    img = img.resize((SIZE, SIZE), Image.LANCZOS)
    rgb = Image.merge("RGB", [img, img, img])
    rgb.save(out_img)

    with open(out_txt, "w") as f:
        f.write(CAPTION)

    if i % 50 == 0:
        print(f"  {i+1}/{len(src_files)}")

print(f"Done. {len(src_files)} images written to {OUT_DIR}")

## Cell 4 — Verify images exist
Should show ~1852 items (926 images + 926 txt files).

In [ ]:
import os

BASE = "/content/drive/MyDrive/spleen_project"

for path in [
    f"{BASE}/lora/train_data",
    f"{BASE}/lora/train_data/10_ultrasound",
]:
    if os.path.exists(path):
        contents = os.listdir(path)
        print(f"EXISTS: {path}")
        print(f"  {len(contents)} items — first 3: {sorted(contents)[:3]}")
    else:
        print(f"MISSING: {path}")

## Cell 5 — Run LoRA training

**Before running:** paste this in your browser console (F12) to prevent disconnection:
```javascript
function ClickConnect() {
  console.log("Keeping alive...");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000);
```

**Key flags:**
- `--train_data_dir` — points to the *parent* of `10_ultrasound/`, not inside it
- `--network_dim=16` — LoRA rank (higher = more capacity, more VRAM)
- `--resolution="512,512"` — must be a CLI arg, not just in config
- `--resume_from_checkpoint=latest` — safe to re-run if disconnected

In [ ]:
%%bash
cd /content/sd-scripts

BASE="/content/drive/MyDrive/spleen_project"

accelerate launch \
  --num_cpu_threads_per_process 2 \
  train_network.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="$BASE/lora/train_data" \
  --output_dir="$BASE/lora/output" \
  --output_name="spleen_lora" \
  --network_module="networks.lora" \
  --network_dim=16 \
  --network_alpha=8 \
  --optimizer_type="AdamW8bit" \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --lr_scheduler="cosine_with_restarts" \
  --lr_warmup_steps=100 \
  --lr_scheduler_num_cycles=2 \
  --resolution="512,512" \
  --mixed_precision="fp16" \
  --train_batch_size=2 \
  --gradient_accumulation_steps=2 \
  --max_train_steps=5000 \
  --save_every_n_steps=500 \
  --mixed_precision="fp16" \
  --save_precision="fp16" \
  --noise_offset=0.1 \
  --caption_extension=".txt" \
  --xformers \
  --enable_bucket \
  2>&1 | tee "$BASE/lora/logs/train_log.txt"

## Cell 6 — Verify LoRA output
You should see `spleen_lora.safetensors` (~70 MB). That's the final trained model.

In [ ]:
import os

BASE    = "/content/drive/MyDrive/spleen_project"
out_dir = f"{BASE}/lora/output"

print("Files in lora/output/:")
for f in sorted(os.listdir(out_dir)):
    size_mb = os.path.getsize(os.path.join(out_dir, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

# Expected:
#   spleen_lora-000500.safetensors  <- intermediate checkpoints
#   spleen_lora-001000.safetensors
#   ...
#   spleen_lora.safetensors         <- final model (~70 MB)